# Practical 4 — HR Analytics (People Charm Case)

## Objective
Inspect missing values, third quartile of last evaluation, department-by-salary crosstab, median number of projects, monthly-hours histogram, and boxplots.

**How to use:** Put the required CSV file(s) in the `data/` folder, update the filename/path if needed, and run cells from top to bottom. Where a dataset is unavailable, the notebook reports that clearly rather than inventing results.

## Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
csv_path = DATA_DIR / "People Charm case.csv"
if not csv_path.exists():
    raise FileNotFoundError("Place 'People Charm case.csv' in the data/ folder.")
hr = pd.read_csv(csv_path)
display(hr.head())
print("Columns:", hr.columns.tolist())

In [ ]:
print("Columns containing missing values:")
display(hr.isna().sum().loc[lambda s: s > 0].to_frame("missing_count"))

def find_col(df, candidates):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in lookup:
            return lookup[name.lower()]
    raise KeyError(f"Could not find columns {candidates}; available: {list(df.columns)}")

eval_col = find_col(hr, ["lastEvaluvation", "lastEvaluation", "last_evaluation"])
dept_col = find_col(hr, ["dept", "department"])
salary_col = find_col(hr, ["salary"])
projects_col = find_col(hr, ["numberOfProjects", "number_project", "number_of_projects"])
hours_col = find_col(hr, ["avgMonthlyHours", "average_montly_hours", "avg_monthly_hours"])

In [ ]:
evaluation = pd.to_numeric(hr[eval_col], errors="coerce")
print("Third quartile of last evaluation:", evaluation.quantile(0.75))
cross = pd.crosstab(hr[dept_col], hr[salary_col], dropna=False)
display(cross)
if "low" in cross.columns:
    print("Department with highest frequency in low salary:", cross["low"].idxmax())
else:
    print("No salary category exactly named 'low'. Check the crosstab column labels above.")
projects = pd.to_numeric(hr[projects_col], errors="coerce")
print("Median number of projects:", projects.median())

In [ ]:
hours = pd.to_numeric(hr[hours_col], errors="coerce").dropna()
plt.figure(figsize=(8, 5))
counts, edges, _ = plt.hist(hours, bins=20, edgecolor="black")
plt.xlabel("Average monthly hours")
plt.ylabel("Number of employees")
plt.title("Average monthly hours distribution")
plt.tight_layout()
plt.show()
for i, (left, right, count) in enumerate(zip(edges[:-1], edges[1:], counts)):
    if left <= 150 <= right:
        print(f"The bin containing 150 hours is approximately [{left:.2f}, {right:.2f}] with {int(count)} employees.")
        break

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].boxplot(pd.to_numeric(hr[eval_col], errors="coerce").dropna())
axes[0].set_title("Last evaluation")
axes[0].set_ylabel("Score")
axes[1].boxplot(pd.to_numeric(hr[projects_col], errors="coerce").dropna())
axes[1].set_title("Number of projects")
axes[1].set_ylabel("Projects")
plt.tight_layout()
plt.show()

## Conclusion

Summarize the missing columns, quartile, low-salary department, median project count, and the histogram interval containing 150 hours.